<a href="https://colab.research.google.com/github/shellyycao/IDS705_ML_Final_Project_Group10/blob/Arvind/corrupted_images_script_fixed.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Creating corrupted images


In [ ]:
# ─────────────────────────────────────────────────────────────
# Grid-by-Grid Image Corruption Script — Group 10 IDS 705
# Sebine Scaria
# 224x224 PneumoniaMNIST
# ─────────────────────────────────────────────────────────────

# ── Step 1: Mount Drive and install ──────────────────────────
import shutil

# Clear the mount point first
shutil.rmtree('/content/drive', ignore_errors=True)

# Now remount
from google.colab import drive
drive.mount('/content/drive')

!pip install medmnist --quiet

import torch
import torch.nn.functional as F
import torchvision.transforms as transforms
import torchvision.transforms.functional as TF
import numpy as np
import os
import io
from PIL import Image
from medmnist import INFO, PneumoniaMNIST
from torch.utils.data import DataLoader

print('Libraries loaded!')

Mounted at /content/drive
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 14.9 MB/s eta 0:00:00
Libraries loaded!


In [ ]:
# ── Step 2: Save paths ────────────────────────────────────────
# ── Run this every time Colab resets ─────────────────────────
BASE_DIR      = '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)'
CLEAN_DIR     = os.path.join(BASE_DIR, 'Clean_images')
CORRUPTED_DIR = '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images'

os.makedirs(CORRUPTED_DIR, exist_ok=True)

print(f'Clean dir:     {CLEAN_DIR}')
print(f'Corrupted dir: {CORRUPTED_DIR}')

Clean dir:     /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Clean_images
Corrupted dir: /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images


In [ ]:
# ── Step 3: Load datasets ─────────────────────────────────────
SIZE = 224

data_transform = transforms.Compose([
    transforms.Resize((SIZE, SIZE), interpolation=Image.NEAREST),
    transforms.Lambda(lambda img: img.convert('L')),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

train_dataset = PneumoniaMNIST(split='train', transform=data_transform, download=True, size=SIZE)
val_dataset   = PneumoniaMNIST(split='val',   transform=data_transform, download=True, size=SIZE)
test_dataset  = PneumoniaMNIST(split='test',  transform=data_transform, download=True, size=SIZE)

def load_all(dataset):
    loader = DataLoader(dataset, batch_size=len(dataset), shuffle=False)
    images, labels = next(iter(loader))
    return images, labels

train_images, train_labels = load_all(train_dataset)
val_images,   val_labels   = load_all(val_dataset)
test_images,  test_labels  = load_all(test_dataset)

print(f'Train: {train_images.shape}')
print(f'Val:   {val_images.shape}')
print(f'Test:  {test_images.shape}')

100%|██████████| 214M/214M [00:12<00:00, 16.6MB/s]


Train: torch.Size([4708, 1, 224, 224])
Val:   torch.Size([524, 1, 224, 224])
Test:  torch.Size([624, 1, 224, 224])


In [ ]:
# ── Step 4: Attack functions at L3 ───────────────────────────

def attack_brightness_contrast(region):
    """L3: Brightness dark delta=-0.3, Contrast low alpha=0.4"""
    region = torch.clamp(region - 0.3, -1, 1)
    region = torch.clamp(region * 0.4, -1, 1)
    return region

def attack_jpeg(region):
    """L3: JPEG quality=30"""
    results = []
    for img in region:
        img_01     = (img * 0.5 + 0.5).clamp(0, 1)
        img_pil    = TF.to_pil_image(img_01.squeeze(0))
        buf        = io.BytesIO()
        img_pil.save(buf, format='JPEG', quality=30)
        buf.seek(0)
        img_back   = Image.open(buf).copy()
        img_tensor = TF.to_tensor(img_back).unsqueeze(0)
        results.append((img_tensor - 0.5) / 0.5)
    return torch.cat(results, dim=0)

def attack_downsampling(region):
    """L3: scale=0.3"""
    _, _, H, W = region.shape
    small_h    = max(1, int(H * 0.3))
    small_w    = max(1, int(W * 0.3))
    down = F.interpolate(region, size=(small_h, small_w), mode='nearest')
    return F.interpolate(down, size=(H, W), mode='nearest')

def attack_gaussian_blur(region):
    """L3: kernel=11, sigma=2.0"""
    return TF.gaussian_blur(region, kernel_size=11, sigma=2.0)

ATTACK_NAMES = [
    'brightness_contrast',
    'jpeg_compression',
    'downsampling',
    'gaussian_blur'
]
print(f'4 attacks defined at L3: {ATTACK_NAMES}')

4 attacks defined at L3: ['brightness_contrast', 'jpeg_compression', 'downsampling', 'gaussian_blur']


In [ ]:
# ── Step 5: Grid bounds helper ────────────────────────────────
def get_grid_bounds(H, W, row, col):
    h_start = row * H // 3
    h_end   = (row + 1) * H // 3
    w_start = col * W // 3
    w_end   = (col + 1) * W // 3
    return h_start, h_end, w_start, w_end

In [ ]:
# ── Step 6: Generate and save in chunks (truly low memory) ────
CHUNK_SIZE = 200  # original images per chunk (~1.4 GB of corrupted data each)

def generate_and_save_chunks(dataset, split_name, save_dir):
    """
    Processes CHUNK_SIZE original images at a time, producing
    CHUNK_SIZE × 9 grid positions × 4 attacks = CHUNK_SIZE×36
    corrupted images per chunk, and saves each chunk immediately
    to Drive before moving on. This keeps RAM usage bounded.
    """
    print(f'\nCorrupting {split_name} set in chunks of {CHUNK_SIZE} images...')
    N = len(dataset)

    attack_fns = [
        attack_brightness_contrast,
        attack_jpeg,
        attack_downsampling,
        attack_gaussian_blur
    ]

    chunk_idx = 0
    chunk_paths = []

    for chunk_start in range(0, N, CHUNK_SIZE):
        chunk_end = min(chunk_start + CHUNK_SIZE, N)
        print(f'  Chunk {chunk_idx}: images {chunk_start}–{chunk_end-1}...')

        all_corrupted  = []
        all_labels     = []
        all_attack_ids = []
        all_grid_pos   = []

        for i in range(chunk_start, chunk_end):
            img, label = dataset[i]
            img = img.unsqueeze(0)
            H, W = img.shape[2], img.shape[3]

            for row in range(3):
                for col in range(3):
                    h_start, h_end, w_start, w_end = get_grid_bounds(H, W, row, col)
                    for attack_id, attack_fn in enumerate(attack_fns):
                        corrupted = img.clone()
                        corrupted[:, :, h_start:h_end, w_start:w_end] = attack_fn(
                            img[:, :, h_start:h_end, w_start:w_end]
                        )
                        all_corrupted.append(corrupted)
                        all_labels.append(torch.tensor(label).unsqueeze(0))
                        all_attack_ids.append(attack_id)
                        all_grid_pos.append((row, col))

        corrupted_images = torch.cat(all_corrupted, dim=0)
        corrupted_labels = torch.cat(all_labels,    dim=0)
        attack_ids       = torch.tensor(all_attack_ids)

        chunk_path = os.path.join(save_dir, f'{split_name}_corrupted_chunk{chunk_idx:03d}.pt')
        torch.save({
            'images'        : corrupted_images,
            'labels'        : corrupted_labels,
            'attack_ids'    : attack_ids,
            'attack_names'  : ATTACK_NAMES,
            'grid_positions': all_grid_pos,
            'split'         : split_name,
            'size'          : SIZE,
            'severity'      : 'L3',
            'chunk_idx'     : chunk_idx,
            'orig_range'    : (chunk_start, chunk_end),
            'note'          : f'Grid-corrupted 224x224 {split_name} chunk {chunk_idx} — 4 attacks x 9 grid positions L3'
        }, chunk_path)

        chunk_paths.append(chunk_path)
        chunk_size_mb = os.path.getsize(chunk_path) / (1024 * 1024)
        print(f'    ✅ Saved {corrupted_images.shape[0]:,} images → {chunk_path} ({chunk_size_mb:.0f} MB)')

        # Free memory before next chunk
        del all_corrupted, all_labels, corrupted_images, corrupted_labels, attack_ids
        chunk_idx += 1

    # Save a manifest so the training script knows which chunks to load
    manifest_path = os.path.join(save_dir, f'{split_name}_manifest.txt')
    with open(manifest_path, 'w') as f:
        for p in chunk_paths:
            f.write(p + '\n')
    print(f'  ✅ {split_name} complete: {chunk_idx} chunks saved.')
    print(f'  Manifest: {manifest_path}')
    return chunk_paths

# ── Run train ─────────────────────────────────────────────────
generate_and_save_chunks(train_dataset, 'train', CORRUPTED_DIR)



Corrupting train set in chunks of 200 images...
  Chunk 0: images 0–199...
    ✅ Saved 7,200 images → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk000.pt (1378 MB)
  Chunk 1: images 200–399...
    ✅ Saved 7,200 images → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk001.pt (1378 MB)
  Chunk 2: images 400–599...
    ✅ Saved 7,200 images → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk002.pt (1378 MB)
  Chunk 3: images 600–799...
    ✅ Saved 7,200 images → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk003.pt (1378 MB)
  Chunk 4: images 800–999...
    ✅ Saved 7,200 images → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk004.pt (1378 MB)
  Chunk 5: images 1000–

['/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk000.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk001.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk002.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk003.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk004.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk005.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk006.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/train_corrupted_chunk007.pt',
 '/conte

In [ ]:
generate_and_save_chunks(val_dataset, 'val', CORRUPTED_DIR)



Corrupting val set in chunks of 200 images...
  Chunk 0: images 0–199...
    ✅ Saved 7,200 images → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/val_corrupted_chunk000.pt (1378 MB)
  Chunk 1: images 200–399...
    ✅ Saved 7,200 images → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/val_corrupted_chunk001.pt (1378 MB)
  Chunk 2: images 400–523...
    ✅ Saved 4,464 images → /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/val_corrupted_chunk002.pt (855 MB)
  ✅ val complete: 3 chunks saved.
  Manifest: /content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/val_manifest.txt


['/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/val_corrupted_chunk000.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/val_corrupted_chunk001.pt',
 '/content/drive/MyDrive/Clean_images_and_corrupted_images_(by grid)/Corrupted_by_grid_images/val_corrupted_chunk002.pt']

In [ ]:
# ── Step 7: Verify all chunks landed on Drive ────────────────
import os

print('=== Files saved to Drive ===')
for fname in sorted(os.listdir(CORRUPTED_DIR)):
    fpath = os.path.join(CORRUPTED_DIR, fname)
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    print(f'  {fname:<45s}  {size_mb:6.0f} MB')

print('\n=== Manifests ===')
for split in ['train', 'val']:
    manifest = os.path.join(CORRUPTED_DIR, f'{split}_manifest.txt')
    if os.path.exists(manifest):
        with open(manifest) as mf:
            lines = mf.readlines()
        print(f'  {split}: {len(lines)} chunks')
    else:
        print(f'  {split}: manifest NOT FOUND')


=== Files saved to Drive ===
  train_corrupted_chunk000.pt                      1378 MB
  train_corrupted_chunk001.pt                      1378 MB
  train_corrupted_chunk002.pt                      1378 MB
  train_corrupted_chunk003.pt                      1378 MB
  train_corrupted_chunk004.pt                      1378 MB
  train_corrupted_chunk005.pt                      1378 MB
  train_corrupted_chunk006.pt                      1378 MB
  train_corrupted_chunk007.pt                      1378 MB
  train_corrupted_chunk008.pt                      1378 MB
  train_corrupted_chunk009.pt                      1378 MB
  train_corrupted_chunk010.pt                      1378 MB
  train_corrupted_chunk011.pt                      1378 MB
  train_corrupted_chunk012.pt                      1378 MB
  train_corrupted_chunk013.pt                      1378 MB
  train_corrupted_chunk014.pt                      1378 MB
  train_corrupted_chunk015.pt                      1378 MB
  train_corrupted_chunk016.